In [2]:
!pip install ultralytics albumentations opencv-python tqdm matplotlib -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 78.8 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls /content/drive/MyDrive/mes_project

augmented_labels  KITTI_ORIG   MES_DATASET_aug
KITTI_AUG	  MES_DATASET  original_labels


In [ ]:
import os

WORK = "/content/drive/MyDrive/mes_project_work"

os.makedirs(WORK, exist_ok=True)

print("Working directory created")

Working directory created


In [ ]:
import os
BASE = "/content/drive/MyDrive/mes_project_work"

ORIG_IMG = "/content/drive/MyDrive/mes_project/MES_DATASET"
AUG_IMG  = "/content/drive/MyDrive/mes_project/MES_DATASET_aug"

ORIG_LABEL = "/content/drive/MyDrive/mes_project/original_labels"
AUG_LABEL  = "/content/drive/MyDrive/mes_project/augmented_labels"

KITTI_ORIG = f"{BASE}/KITTI_ORIG"
KITTI_AUG  = f"{BASE}/KITTI_AUG"

In [13]:
classes = {
0:"Car",
1:"Motorcycle",
2:"Bicycle",
3:"Pedestrian",
4:"Auto",
5:"Bus",
6:"Truck",
7:"Tractor",
8:"Animal"
}

In [ ]:
vehicle_map = {
2:0,    # car → Car
3:1,    # motorcycle → Motorcycle
1:2,    # bicycle → Bicycle
0:3,    # person → Pedestrian
5:5,    # bus → Bus
7:6     # truck → Truck
}

animal_classes = [15,16,17,18,19,20,21,22,23,24]

In [ ]:
for folder in [KITTI_ORIG, KITTI_AUG]:
    os.makedirs(folder, exist_ok=True)

## Copying image to original

In [ ]:
import shutil
import os

orig_images = "/content/drive/MyDrive/mes_project/MES_DATASET"
image_folder = os.path.join("/content/drive/MyDrive/mes_project/KITTI_ORIG/training", "image_2")

os.makedirs(image_folder, exist_ok=True)

images = sorted([
    f for f in os.listdir(orig_images)
    if f.lower().endswith(('.png', '.jpg', '.jpeg'))
])

for i, img in enumerate(images):
    src = os.path.join(orig_images, img)
    dst = os.path.join(image_folder, f"{i:06d}.png")

    if not os.path.exists(src):
        print(f"Missing: {src}")
        continue

    shutil.copy(src, dst)

print("Images copied:", len(images))

Images copied: 1118


##Setting up path

In [4]:
import os

BASE = "/content/drive/MyDrive/mes_project"

IMAGE_FOLDER = f"{BASE}/KITTI_ORIG/training/image_2"
YOLO_LABELS = "/content/drive/MyDrive/mes_project/original_labels"

VELODYNE_FOLDER = f"{BASE}/KITTI_ORIG/training/velodyne"
LABEL_FOLDER = f"{BASE}/KITTI_ORIG/training/label_2"
CALIB_FOLDER = f"{BASE}/KITTI_ORIG/training/calib"

for f in [VELODYNE_FOLDER, LABEL_FOLDER, CALIB_FOLDER]:
    os.makedirs(f, exist_ok=True)

##importing midas


In [9]:
import torch
import cv2
import numpy as np
from ultralytics import YOLO
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

midas = torch.hub.load("intel-isl/MiDaS","DPT_Hybrid")
midas.to(device)
midas.eval()

transforms = torch.hub.load("intel-isl/MiDaS","transforms")
transform = transforms.dpt_transform

Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master
Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master


In [8]:
sample = cv2.imread(os.path.join(IMAGE_FOLDER,sorted(os.listdir(IMAGE_FOLDER))[0]))

h,w,_ = sample.shape

fx,fy = 700.0,700.0
cx,cy = w/2.0,h/2.0

##velodyne generation

In [ ]:
from tqdm import tqdm

images = sorted(os.listdir(IMAGE_FOLDER))

for img_name in tqdm(images):

    idx = int(img_name.split(".")[0])

    vel_file = os.path.join(VELODYNE_FOLDER,f"{idx:06d}.bin")

    if os.path.exists(vel_file):
        continue

    img = cv2.imread(os.path.join(IMAGE_FOLDER,img_name))
    img_rgb = cv2.cvtColor(img,cv2.COLOR_BGR2RGB)

    input_batch = transform(img_rgb).to(device)

    with torch.no_grad():

        prediction = midas(input_batch)

        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=img_rgb.shape[:2],
            mode="bicubic",
            align_corners=False
        ).squeeze()

    depth = prediction.cpu().numpy()

    depth = (depth-depth.min())/(depth.max()-depth.min()+1e-8)

    h,w = depth.shape

    u,v = np.meshgrid(np.arange(w),np.arange(h))

    Z = depth * 50
    X = (u-cx) * Z / fx
    Y = (v-cy) * Z / fy

    points = np.stack((X,Y,Z),axis=-1).reshape(-1,3)

    points = points[(points[:,2] > 1.5) & (points[:,2] < 60)]

    keep = int(len(points)*0.03)
    points = points[np.random.choice(len(points),keep,replace=False)]

    intensity = np.random.uniform(0.2,1.0,(points.shape[0],1))

    points = np.concatenate((points,intensity),axis=1).astype(np.float32)

    points.tofile(vel_file)

print("Velodyne generation finished")

100%|██████████| 1118/1118 [13:17<00:00,  1.40it/s]

Velodyne generation finished


##LABELS GENERATION

In [16]:
def estimate_box_from_depth(depth,box):

    x1,y1,x2,y2 = map(int,box)

    crop = depth[y1:y2,x1:x2]

    if crop.size == 0:
        return 1.5,1.6,3.7,0,0,20

    h,w = crop.shape

    u,v = np.meshgrid(np.arange(w),np.arange(h))

    Z = crop * 50

    X = (u + x1 - cx) * Z / fx
    Y = (v + y1 - cy) * Z / fy

    pts = np.stack((X,Y,Z),axis=-1).reshape(-1,3)

    pts = pts[(pts[:,2] > 1.5) & (pts[:,2] < 60)]

    if len(pts) == 0:
        return 1.5,1.6,3.7,0,0,20

    xmin,xmax = pts[:,0].min(),pts[:,0].max()
    ymin,ymax = pts[:,1].min(),pts[:,1].max()
    zmin,zmax = pts[:,2].min(),pts[:,2].max()

    length = xmax-xmin
    width = ymax-ymin
    height = zmax-zmin

    xc = (xmin+xmax)/2
    yc = (ymin+ymax)/2
    zc = (zmin+zmax)/2

    return height,width,length,xc,yc,zc

In [17]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

midas = torch.hub.load("intel-isl/MiDaS", "DPT_Hybrid")
midas.to(device)
midas.eval()

transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
transform = transforms.dpt_transform

Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master
Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master


In [18]:
import numpy as np
import cv2
import os
import torch
from ultralytics import YOLO
from tqdm import tqdm

model = YOLO("yolov8m.pt")

class_map = {
0:"Car",
1:"Motorcycle",
2:"Bicycle",
3:"Pedestrian",
4:"Auto",
5:"Bus",
6:"Truck",
7:"Tractor",
8:"Animal"
}

fx, fy = 700.0, 700.0
cx, cy = 640.0, 360.0

images = sorted(os.listdir(IMAGE_FOLDER))

BATCH_SIZE = 8   # Faster YOLO inference

for start in tqdm(range(0, len(images), BATCH_SIZE)):

    batch = images[start:start+BATCH_SIZE]
    paths = [os.path.join(IMAGE_FOLDER, img) for img in batch]

    results = model(paths, verbose=False)

    for img_name, result in zip(batch, results):

        idx = int(img_name.split(".")[0])
        label_file = os.path.join(LABEL_FOLDER, f"{idx:06d}.txt")

        # Skip already processed images
        if os.path.exists(label_file):
            continue

        img = cv2.imread(os.path.join(IMAGE_FOLDER, img_name))

        if img is None:
            continue

        h_img, w_img, _ = img.shape

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # -------- MiDaS depth --------
        input_batch = transform(img_rgb).to(device)

        with torch.no_grad():

            prediction = midas(input_batch)

            prediction = torch.nn.functional.interpolate(
                prediction.unsqueeze(1),
                size=img_rgb.shape[:2],
                mode="bicubic",
                align_corners=False
            ).squeeze()

        depth = prediction.cpu().numpy()
        depth = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)

        boxes = result.boxes.xyxy.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy()

        with open(label_file, "w") as f:

            for box, cls in zip(boxes, classes):

                cls = int(cls)

                if cls not in class_map:
                    continue

                x1, y1, x2, y2 = box

                x1 =max(0, int(x1))
                y1 =max(0, int(y1))
                x2 =min(w_img-1, int(x2))
                y2 =min(h_img-1, int(y2))

                crop = depth[y1:y2, x1:x2]

                if crop.size == 0:
                    continue

                z = np.median(crop) * 50

                xc = ((x1+x2)/2-cx)*z/fx
                yc = ((y1+y2)/2-cy)*z/fy

                height =1.5
                width  =1.6
                length =3.7
                rot =0
                obj_type =class_map[cls]
                f.write(
                    f"{obj_type} 0 0 0 "
                    f"{x1:.2f} {y1:.2f} {x2:.2f} {y2:.2f} "
                    f"{height:.2f} {width:.2f} {length:.2f} "
                    f"{xc:.2f} {yc:.2f} {z:.2f} {rot:.2f}\n"
                )

print("KITTI labels generation resumed and completed")

100%|██████████| 140/140 [07:10<00:00,  3.07s/it]

KITTI labels generation resumed and completed


In [20]:
import os

files = sorted(os.listdir(LABEL_FOLDER))
print("Total labels:", len(files))
print("Last label:", files[-1])

Total labels: 1118
Last label: 001117.txt


In [21]:
# ============================================================
# LABEL STATISTICS FOR KITTI 3D LABELS
# ============================================================

import os
from collections import defaultdict

# ============================================================
# PATH (CHANGE IF NEEDED)
# ============================================================

label_dir = "/content/drive/MyDrive/mes_project/KITTI_ORIG/training/label_2"   # change for AUG later

# ============================================================
# STORAGE
# ============================================================

object_counts = defaultdict(int)
image_counts = defaultdict(int)

total_images = 0
total_objects = 0

# ============================================================
# PROCESS FILES
# ============================================================

for file in os.listdir(label_dir):

    if not file.endswith(".txt"):
        continue

    total_images += 1

    file_path = os.path.join(label_dir, file)

    with open(file_path) as f:
        lines = f.readlines()

    classes_in_image = set()

    for line in lines:

        parts = line.strip().split()

        if len(parts) < 15:
            continue

        cls = parts[0]   # class name (Car, Truck, etc.)

        object_counts[cls] += 1
        total_objects += 1

        classes_in_image.add(cls)

    for cls in classes_in_image:
        image_counts[cls] += 1

# ============================================================
# PRINT RESULTS
# ============================================================

print("\n==============================")
print("LABEL STATISTICS")
print("==============================")

print(f"\nTotal Images: {total_images}")
print(f"Total Objects: {total_objects}")
print(f"Average Objects per Image: {total_objects / total_images:.2f}")

print("\nOBJECT COUNTS PER CLASS:")
for cls, count in sorted(object_counts.items(), key=lambda x: -x[1]):
    print(f"{cls}: {count}")

print("\nIMAGE COUNTS PER CLASS:")
for cls, count in sorted(image_counts.items(), key=lambda x: -x[1]):
    print(f"{cls}: {count}")

print("\n==============================")


LABEL STATISTICS

Total Images: 1118
Total Objects: 7550
Average Objects per Image: 6.75

OBJECT COUNTS PER CLASS:
Car: 3220
Bicycle: 1673
Tractor: 1077
Pedestrian: 1012
Bus: 327
Motorcycle: 232
Truck: 6
Animal: 2
Auto: 1

IMAGE COUNTS PER CLASS:
Car: 844
Tractor: 681
Bicycle: 676
Pedestrian: 500
Bus: 259
Motorcycle: 156
Truck: 6
Animal: 2
Auto: 1



##Calib Generation Script

In [ ]:
from tqdm import tqdm
import cv2
import os

sample_img = cv2.imread(os.path.join(IMAGE_FOLDER, sorted(os.listdir(IMAGE_FOLDER))[0]))
h, w, _ = sample_img.shape

fx, fy = 700.0, 700.0
cx, cy = w / 2.0, h / 2.0

images = sorted(os.listdir(IMAGE_FOLDER))

for img_name in tqdm(images):

    idx = int(img_name.split(".")[0])
    calib_file = os.path.join(CALIB_FOLDER, f"{idx:06d}.txt")

    if os.path.exists(calib_file):
        continue

    with open(calib_file, "w") as f:
        f.write(f"P2: {fx} 0 {cx} 0 0 {fy} {cy} 0 0 0 1 0\n")
        f.write("R0_rect: 1 0 0 0 1 0 0 0 1\n")
        f.write("Tr_velo_to_cam: 1 0 0 0 0 1 0 0 0 0 1 0\n")

print("Calibration files generated ✅")

100%|██████████| 1118/1118 [00:11<00:00, 94.03it/s]

Calibration files generated ✅


In [ ]:
import os

files = sorted(os.listdir(CALIB_FOLDER))
print("Total calibs:", len(files))
print("Last calib:", files[-1])

Total calibs: 1118
Last calib: 001117.txt
